# Yet New — fine-tune the CAD layer labeler (Google Colab or Kaggle, free T4 GPU)

Trains a LoRA adapter on **Qwen2.5-3B-Instruct** (configurable) to answer one label per CAD layer, evaluates it on the held-out venues, and exports a **GGUF q4_k_m** file for Ollama.

**Before you run this**
- Use a real export only: `python scripts/export_dataset.py --out data/dataset`, then zip the folder (`train.jsonl`, `val.jsonl`, `manifest.json`). The notebook refuses the SYNTHETIC dry run.
- Have enough confirmed data first (see `ml/README.md`): 15–30 varied venues, 500+ labeled layers, 20+ per label.
- **Kaggle** (recommended: 30 free GPU hours a week, runs up to 12 h with the browser closed): upload the zip as a **private** dataset (**+ Add Input → Upload**), Settings → Accelerator **GPU T4 x2**, Internet **on**, then **Save Version → Save & Run All (Commit)**. The files appear under the version's **Output**. Customer data: keep the dataset private and delete it after training.
- **Colab:** Runtime → Change runtime type → **T4 GPU**, Run all, pick the zip when asked. `SAVE_TO_DRIVE = True` keeps progress in Google Drive, so Run all after a cut-off continues from the last saved step.

**Smoke mode** (`SMOKE_TEST = True` below) only checks that this Colab environment works: a few training steps on the **synthetic dry-run** export (`export_dataset.py --synthetic-dry-run`), nothing else. Its output is named `SMOKE-TEST-delete-me`, nothing is exported or downloaded, the service and `evaluate.py` refuse any model with "smoke" in its name — and the last cell tells you to delete it.

**What is checked here, and what isn't**
- Package versions were pinned on 2026-10-10 from PyPI metadata: `unsloth 2026.9.11` declares `trl ≤ 0.24.0`, `transformers ≤ 5.5.0`, `datasets < 4.4.0`, `torch < 2.13`. The APIs used below (`FastLanguageModel.from_pretrained/get_peft_model/for_inference`, `train_on_responses_only`, `save_pretrained_gguf(quantization_method="q4_k_m")`, TRL 0.24 `SFTConfig(max_length, dataset_text_field)` and `SFTTrainer(processing_class=…)`, Transformers 5.5 `warmup_steps` as a ratio) were read from those exact wheels.
- **Not verified:** resuming from a checkpoint and the Kaggle path (the assistant notebook's Kaggle run worked, 2026-10-19). This notebook has not been run (no GPU where it was written), so install time, memory use on a T4, training time and the exact GGUF file name Unsloth writes are unconfirmed. Colab's preinstalled torch may be newer than 2.13; pip will then install an older torch (slow, but it works).

The prompts are **not** built here: every row in `train.jsonl` was made by `prompt.training_example` — the same code the service uses at inference.

In [ ]:
# Pinned 2026-10-10 (see the note above). gguf is only used to check the exported file.
!pip install -q "unsloth==2026.9.11" "unsloth_zoo==2026.9.7" "trl==0.24.0" "transformers==5.5.0" "peft==0.21.0" \
    "datasets==4.3.0" "bitsandbytes==0.50.2" "accelerate==1.15.0" "gguf==0.19.0"

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle "T4 x2": train on one GPU

import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU: Colab → Runtime → Change runtime type → T4 GPU; Kaggle → Settings → Accelerator → GPU T4 x2"
print(torch.cuda.get_device_name(0))

## Settings

In [ ]:
# The base must be the same model family the service runs as its baseline in Ollama (qwen2.5:3b).
# 7B (if the inference machine has ~6 GB free RAM): "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
EPOCHS = 2              # 2–3
LEARNING_RATE = 2e-4
LORA_R = 16
MAX_SEQ_LENGTH = 2048   # system prompt with the label guide ≈ 650 tokens + one layer
BATCH_SIZE = 8
GRAD_ACCUM = 2
SEED = 3407

SAVE_TO_DRIVE = False   # Colab: keep checkpoints in Google Drive, so a cut-off session continues where it stopped
SAVE_STEPS = 50         # a checkpoint every N steps

# Smoke mode: verify the Colab environment only (a few steps on the SYNTHETIC dry run).
SMOKE_TEST = False
SMOKE_STEPS = 5

OUTPUT_NAME = "SMOKE-TEST-delete-me" if SMOKE_TEST else "yetnew-layers"
print("SMOKE TEST — environment check only; nothing will be exported." if SMOKE_TEST else f"Real training run → {OUTPUT_NAME}")

## 1. Upload the dataset

In [ ]:
import hashlib, json, pathlib, zipfile

ON_KAGGLE = pathlib.Path("/kaggle/input").exists()
if ON_KAGGLE:
    # + Add Input → the private dataset made from the export zip (Kaggle usually unzips it; if not, it is unzipped here).
    for z in sorted(pathlib.Path("/kaggle/input").rglob("*.zip")):
        zipfile.ZipFile(z).extractall("dataset")
    search = [pathlib.Path("/kaggle/input"), pathlib.Path("dataset")]
else:
    from google.colab import files
    print("Upload the zipped export folder (train.jsonl, val.jsonl, manifest.json).")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("dataset")
    search = [pathlib.Path("dataset")]
found = [p.parent for d in search if d.exists() for p in d.rglob("manifest.json")
         if "venues" in json.loads(p.read_text(encoding="utf-8"))]
assert found, "No layer export found (Kaggle: + Add Input → your export zip dataset)"
root = found[0]
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
DATA_ID = hashlib.sha256((root / "train.jsonl").read_bytes()).hexdigest()[:10]

if SMOKE_TEST:
    assert manifest["synthetic"] is True, "Smoke mode runs on the SYNTHETIC dry-run export only, never on customer data."
else:
    assert manifest["synthetic"] is False, "This is the SYNTHETIC dry run. Never train on it (set SMOKE_TEST = True to check the environment)."
assert len(manifest["venues"]["val"]) >= 1 and len(manifest["venues"]["train"]) >= 2, "Too few venues"
assert not set(manifest["venues"]["train"]) & set(manifest["venues"]["val"]), "A venue is in both splits"

def load(name):
    return [json.loads(line) for line in (root / name).read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows, val_rows = load("train.jsonl"), load("val.jsonl")
print(f"prompt version {manifest['prompt_version']} · text samples {'on' if manifest['include_text_samples'] else 'off'}")
print(f"{len(train_rows)} training rows ({manifest['examples']['train_augmented']} augmented), {len(val_rows)} validation rows")
print("label counts (train / val):")
for label, n in manifest["label_counts"]["train"].items():
    print(f"  {label:<12} {n:>5} {manifest['label_counts']['val'][label]:>5}")
for w in manifest["warnings"]:
    print("WARNING:", w)

## 2. Load the base model and check the chat format

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None,
)

# Training text must be exactly what Ollama sends at inference (ml/yetnew_layers/chat_format.expected).
def expected(messages, add_generation_prompt):
    text = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return text + ("<|im_start|>assistant\n" if add_generation_prompt else "")

def as_text(messages, add_generation_prompt=False):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)

sample = train_rows[0]["messages"]
assert as_text(sample) == expected(sample, False), "The tokenizer's chat template is not the expected Qwen2.5 format"
assert as_text(sample[:2], True) == expected(sample[:2], True)

lengths = [len(tokenizer(as_text(r["messages"]), add_special_tokens=False)["input_ids"]) for r in train_rows + val_rows]
print("longest example:", max(lengths), "tokens")
assert max(lengths) <= MAX_SEQ_LENGTH, "Raise MAX_SEQ_LENGTH"

## 3. LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

## 4. Train (loss on the answer only)

In [ ]:
import glob
from datasets import Dataset

# Checkpoints belong to one run: same data, epochs, rate and LoRA size — anything else starts over.
RUN_ID = f"{OUTPUT_NAME}-{DATA_ID}-e{EPOCHS}-lr{LEARNING_RATE}-r{LORA_R}-s{SEED}"
if SAVE_TO_DRIVE and not ON_KAGGLE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINTS = f"/content/drive/MyDrive/yetnew-checkpoints/{RUN_ID}"
else:
    CHECKPOINTS = f"checkpoints/{RUN_ID}"
saved = sorted(glob.glob(f"{CHECKPOINTS}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = saved[-1] if saved and not SMOKE_TEST else None
print(f"continuing from {resume}" if resume else f"starting fresh · checkpoints in {CHECKPOINTS}")
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

# Each text ends with "<label><|im_end|>"; the final newline is dropped so no extra EOS is appended.
train_ds = Dataset.from_list([{"text": as_text(r["messages"]).removesuffix("\n")} for r in train_rows])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LENGTH, packing=False,
        num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE, lr_scheduler_type="linear", warmup_steps=0.05, weight_decay=0.01,
        optim="adamw_8bit", logging_steps=5, seed=SEED, output_dir=CHECKPOINTS, save_strategy="no" if SMOKE_TEST else "steps", save_steps=SAVE_STEPS, save_total_limit=1, report_to="none",
        max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

# Only the label (and <|im_end|>) may carry loss:
ex = trainer.train_dataset[0]
kept = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
trained, label = tokenizer.decode(kept).strip(), train_rows[0]["label"]
print("trained on:", repr(trained), "| expected:", repr(label + "<|im_end|>"))
# The label, then only end-of-turn tokens (an extra <|im_end|> appended by TRL would be harmless).
assert trained.startswith(label + "<|im_end|>") and not trained[len(label):].replace("<|im_end|>", "").strip()

stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

## 5. Validation (held-out venues)

Indicative only: the comparison that decides which model the service uses is `scripts/evaluate.py`, run through Ollama on your machine.

In [ ]:
from collections import Counter, defaultdict

FastLanguageModel.for_inference(model)

def predict(messages):
    prompt = as_text(messages[:2], add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=8, do_sample=False)
    text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return text.splitlines()[0].strip().lower() if text else ""

if SMOKE_TEST:
    print(f"SMOKE TEST: {SMOKE_STEPS} steps on synthetic data — these numbers mean nothing.")
per_label, mistakes = defaultdict(lambda: [0, 0]), []
for r in (val_rows[:3] if SMOKE_TEST else val_rows):
    got = predict(r["messages"])
    per_label[r["label"]][1] += 1
    if got == r["label"]:
        per_label[r["label"]][0] += 1
    else:
        mistakes.append((r["building_id"], json.loads(r["messages"][1]["content"])["layer"], r["label"], got))

correct, total = sum(c for c, _ in per_label.values()), sum(n for _, n in per_label.values())
print(f"accuracy {correct}/{total} = {correct / total:.1%}")
for label, (c, n) in sorted(per_label.items()):
    print(f"  {label:<12} {c:>3}/{n:<3} {c / n:.0%}")
print("mistakes (venue, layer, expected, got):")
for m in mistakes:
    print("  ", m)

## 6. Export: LoRA adapter + GGUF q4_k_m

In [ ]:
import glob, shutil
from gguf import GGUFReader
from jinja2.sandbox import ImmutableSandboxedEnvironment

if SMOKE_TEST:
    raise SystemExit(
        "SMOKE TEST finished: the environment works. Nothing was exported, on purpose.\n"
        "DELETE IT NOW: Colab → Runtime → Disconnect and delete runtime (Kaggle: delete this version) (removes the SMOKE-TEST-delete-me weights).\n"
        "Then set SMOKE_TEST = False and train on a real export."
    )
assert "smoke" not in OUTPUT_NAME.lower()

model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
shutil.make_archive(f"{OUTPUT_NAME}-lora", "zip", "lora_adapter")

# The merged model and the f16 GGUF are ~12 GB: on Kaggle they go to /tmp, not the 20 GB output folder.
GGUF_DIR = "/tmp/gguf_out" if ON_KAGGLE else "gguf_out"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
found = sorted(p for d in (GGUF_DIR, ".", "/tmp") for p in glob.glob(f"{d}/**/*.gguf", recursive=True) if "q4_k_m" in p.lower())
print("GGUF files:", found)
src = found[0]

# Ollama renders chats with the template embedded in the GGUF: it must render the training text.
embedded = GGUFReader(src).fields["tokenizer.chat_template"].contents()
render = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True).from_string(embedded).render
for r in val_rows[:5] + train_rows[:5]:
    assert render(messages=r["messages"][:2], add_generation_prompt=True, tools=None) == expected(r["messages"][:2], True)
print("Embedded chat template renders the training prompt ✓")

final = f"{OUTPUT_NAME}.Q4_K_M.gguf"
shutil.copy(src, final)
print(final, round(pathlib.Path(final).stat().st_size / 1e9, 2), "GB")
if ON_KAGGLE:
    shutil.rmtree("checkpoints", ignore_errors=True); shutil.rmtree("lora_adapter", ignore_errors=True)
    shutil.rmtree("dataset", ignore_errors=True)   # customer data: not kept in the version's output
    print(f"Done. Download {final} and {OUTPUT_NAME}-lora.zip from this version's Output tab, then delete the dataset.")
else:
    files.download(f"{OUTPUT_NAME}-lora.zip")
    files.download(final)   # ~2 GB for 3B; if the browser download fails, copy it to Google Drive instead

## 7. On your machine

```powershell
# put yetnew-layers.Q4_K_M.gguf in ml/ollama/ (next to the Modelfile), then from ml/:
ollama create yetnew-layers -f ollama/Modelfile
.venv\Scripts\python.exe scripts\check_template.py --model yetnew-layers   # REQUIRED after every ollama create / Ollama upgrade
.venv\Scripts\python.exe scripts\evaluate.py --dataset data\dataset --write  # beats the baseline on held-out venues?
```

The service switches to the fine-tuned model only if `evaluate.py` wrote a `model_choice.json` in which it won.